## Setup

In [4]:
from dotenv import load_dotenv
from utils import chat_interface
import os

In [5]:
load_dotenv()

True

## Run

In [ ]:
# TODO: Develop your agents under `agentic/agents`
# TODO: Develop your tools under `agentic/tools`
# TODO: Modify `agentic/workflow` in order to orchestrate your agents

In [8]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

In [ ]:
await chat_interface(orchestrator, "1")

User: Hi
Assistant: Hello! How can I assist you today?
User: q
Assistant: Goodbye!


In [5]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='43c56100-7a4e-4ff0-adb7-1fbbfdac82e3'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 60, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C1loRWd5jRqktu5Fut6YZAZsTdc6S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0ab48c4a-6130-4dc9-96cd-271e2be7b7c8-0', usage_metadata={'input_tokens': 60, 'output_tokens': 10, 'total_tokens': 70, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

## Routing Tests (Sample Tickets)

In [ ]:
from uuid import uuid4
from langchain_core.messages import HumanMessage

sample_tickets = [
    {
        "name": "KB hit -> resolver",
        "latest_message": "How do I cancel my reservation?",
        "channel": "chat",
        "main_issue_type": "reservation",
        "tags": "reservation, cancellation",
    },
    {
        "name": "KB miss -> escalation",
        "latest_message": "ZXQ-91 edge-protocol mismatch in latent socket handshake",
        "channel": "chat",
        "main_issue_type": "general",
        "tags": "unknown",
    },
    {
        "name": "Sensitive type -> escalation",
        "latest_message": "There is a fraudulent charge on my account and I need urgent help.",
        "channel": "email",
        "main_issue_type": "fraud",
        "tags": "fraud, billing",
    },
]


def build_ticket_state(sample: dict, thread_id: str) -> dict:
    return {
        "ticket": {
            "ticket_id": thread_id,
            "account_id": "cultpass",
            "user_id": "demo-user",
            "external_user_id": "demo-external-user",
            "channel": sample["channel"],
            "status": "open",
            "main_issue_type": sample.get("main_issue_type"),
            "tags": sample.get("tags"),
            "latest_message": sample["latest_message"],
        },
        "messages": [HumanMessage(content=sample["latest_message"])],
        "short_term_memory": {},
        "long_term_memory": {},
    }


async def demo_routing(orchestrator):
    print("=== Knowledge Retrieval + Escalation Demonstration ===")

    for sample in sample_tickets:
        thread_id = str(uuid4())
        state_input = build_ticket_state(sample, thread_id)
        config = {"configurable": {"thread_id": thread_id}}

        result = await orchestrator.ainvoke(input=state_input, config=config)
        resolution = result.get("resolution") or {}
        classification = result.get("classification") or {}
        retrieved_context = result.get("retrieved_context") or []

        # Drive workflow to terminal state if needed.
        while not resolution and result.get("next_agent") in {"resolver", "escalation", "classifier"}:
            result = await orchestrator.ainvoke(input={"messages": []}, config=config)
            resolution = result.get("resolution") or resolution
            classification = result.get("classification") or classification
            retrieved_context = result.get("retrieved_context") or retrieved_context

        kb_hits = 0
        if retrieved_context and isinstance(retrieved_context[0], dict):
            kb_hits = int(retrieved_context[0].get("total_found", 0) or 0)

        final_route = "escalation" if resolution.get("resolved") is False else "resolver/end"

        print(f"\nCase: {sample['name']}")
        print(f"  Message         : {sample['latest_message']}")
        print(f"  Classified as   : {classification.get('issue_type', 'unknown')}")
        print(f"  Class confidence: {classification.get('confidence', 'n/a')}")
        print(f"  KB hits         : {kb_hits}")
        print(f"  Final route     : {final_route}")
        print(f"  Resolved        : {resolution.get('resolved', 'n/a')}")
        print(f"  Action          : {resolution.get('action_taken', 'n/a')}")


await demo_routing(orchestrator)